# 模块 09：Attention 机制

本 Notebook 从 Scaled Dot-Product Attention 开始，手写 softmax、mask 和 Multi-Head Attention 的核心数据流。

## 1. 理论背景

Attention 的输入是 Query、Key、Value。Query 和 Key 计算相似度，softmax 得到权重，再对 Value 加权求和。

## 1.1 实际作用（用途）
- 让模型按需聚合上下文，是机器翻译、摘要、问答、代码补全和多模态理解的关键机制。
- 能直接连接相距很远的 token，缓解长距离依赖问题。
- Multi-Head Attention 可同时学习语法、指代、局部搭配和全局主题等关系。

## 2. 数学推导

$$Attention(Q,K,V)=softmax(\frac{QK^T}{\sqrt{d_k}})V$$

`sqrt(d_k)` 用来缩放点积，避免维度增大时 softmax 进入饱和区。

In [1]:
import numpy as np

def softmax(x, axis=-1):
    # 减去最大值提升数值稳定性，避免 exp 溢出。
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=axis, keepdims=True)

def scaled_dot_product_attention(q, k, v, mask=None):
    # QK^T 得到每个 token 对其他 token 的匹配分数。
    scores = q @ np.swapaxes(k, -1, -2) / np.sqrt(q.shape[-1])
    if mask is not None:
        # 被 mask 掉的位置设成极小值，softmax 后权重接近 0。
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    # 用注意力权重对 V 加权求和，得到上下文表示。
    return weights @ v, weights


## 3. 代码思路

本章代码直接实现 Scaled Dot-Product Attention 的数据流：

1. `softmax` 先做最大值平移，保证指数运算更稳定。
2. `scaled_dot_product_attention` 用 `QK^T` 计算 token 之间的匹配分数，再除以 `sqrt(d_k)` 控制数值尺度。
3. 如果传入 mask，就把不可见位置替换成极小值，使它们在 softmax 后几乎没有权重。
4. 注意力权重乘以 `V` 得到上下文表示，返回 `context` 和 `weights`，便于同时看结果和注意力分布。
5. 示例中手工构造 Q/K/V，先看普通 attention，再看 causal mask 对权重矩阵的影响。

## 4. NumPy 手写实现

下面用 4 个 token 构造一个最小序列，观察每个 token 对其他 token 的注意力分布。

In [2]:
tokens = ["我", "喜欢", "机器", "学习"]
# 为了便于观察，这里手工构造 Q/K/V；真实模型中它们来自可训练线性层。
q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.5, 1.0]])
k = q.copy()
v = q.copy()

context, weights = scaled_dot_product_attention(q, k, v)
print("attention weights:\n", np.round(weights, 4))
print("context:\n", np.round(context, 4))


attention weights:
 [[0.313  0.1543 0.313  0.2198]
 [0.1412 0.2863 0.2863 0.2863]
 [0.1834 0.1834 0.372  0.2612]
 [0.1626 0.2315 0.3297 0.2763]]
context:
 [[0.7358 0.687 ]
 [0.5706 0.8588]
 [0.686  0.8166]
 [0.6304 0.8374]]


## 5. Mask

自回归生成中，第 `t` 个 token 不能看到未来 token，因此需要 causal mask。

In [3]:
seq_len = len(tokens)
# 下三角 mask 只允许当前位置看见自己和过去位置，常用于自回归生成。
causal_mask = np.tril(np.ones((seq_len, seq_len), dtype=bool))
_, masked_weights = scaled_dot_product_attention(q, k, v, mask=causal_mask)
print(np.round(masked_weights, 4))


[[1.     0.     0.     0.    ]
 [0.3302 0.6698 0.     0.    ]
 [0.2483 0.2483 0.5035 0.    ]
 [0.1626 0.2315 0.3297 0.2763]]


## 6. PyTorch 数值验证

同一组 Q/K/V 用 PyTorch 的基础张量操作计算，确认手写实现一致。

In [4]:
import torch

q_t = torch.tensor(q, dtype=torch.float64)
k_t = torch.tensor(k, dtype=torch.float64)
v_t = torch.tensor(v, dtype=torch.float64)
# 用 PyTorch 复算 attention，确认 NumPy 版本的矩阵维度和 softmax 方向正确。
weights_t = torch.softmax(q_t @ k_t.T / np.sqrt(q.shape[-1]), dim=-1)
context_t = weights_t @ v_t

assert np.allclose(weights, weights_t.numpy(), atol=1e-5)
assert np.allclose(context, context_t.numpy(), atol=1e-5)
print("NumPy attention matches PyTorch")


NumPy attention matches PyTorch
